# CT_scan_processing

**What this notebook showcases in `gyroid_utils`:** the CT-scan analysis use case. It converts a raw TIFF image stack to `.mhd`, loads and processes the CT volume (`CT_scans.py`), extracts an isosurface mesh from the 3D image with marching cubes (`mesh_tools.py`), and produces an interactive HTML visualization of the resulting mesh, including curvature-based coloring (`viz.py`).

**Note:** in the examples provided in this library, there is no `.tiff` stack, but already a `.mhd`/`.raw` volume.

In [1]:
%matplotlib qt
import numpy as np
import tkinter as tk
from tkinter import filedialog
import os
import SimpleITK as sitk

import gyroid_utils
from gyroid_utils.utils import reload_all
reload_all()

working_path = os.getcwd()
print("Current working directory:", working_path)
parent_dir = os.path.dirname(working_path)

[gyroid_utils] version 3.1.4 loaded
[gyroid_utils] version 3.1.4 loaded
gyroid_utils: all modules reloaded
Current working directory: c:\Users\cofo\Documents\02 - GitHub\GYROIDS\notebooks


## 1 — Convert TIFF Stack to MHD

Opens a folder-picker dialog (`tkinter.filedialog.askdirectory`) so you can select the folder containing the raw TIFF slice stack, then converts it into a single `.mhd`/`.raw` volume with `CT_scans.convert_tiff_to_mhd().


In [2]:
# ---- convert tiff to mhd ----
root = tk.Tk()
input_file_path = filedialog.askdirectory() # this is the mhd file contnaing the whole image, no filter, no segmentation
print(input_file_path)
root.withdraw() 

#gyroid_utils.CT_scans.convert_tiff_to_mhd(input_file_path, input_file_path, memory_saver=True)
gyroid_utils.CT_scans.convert_jpg_to_mhd(input_file_path, input_file_path, memory_saver=True)

G:/Limit/COFO/01 PhD research/00 Perso/MRI/AA242789CJS_1.2.276.0.37.1.373.201611.537590_rnr-vuemotion03_VwfuGt31BUe4lVMJCacX5w.1/exam/jpeg
[INFO] gyroid_utils:((convert_jpg_to_mhd)): 139 JPG images found.
[INFO] gyroid_utils:((convert_jpg_to_mhd)): CT scan of dimension (1024, 1024, 139) detected
[INFO] gyroid_utils:((convert_jpg_to_mhd)): Using spacing: (0.2, 0.2, 0.2) and origin: (0.0, 0.0, 0.0)
Progress: [#############################-] 99%

UnidentifiedImageError: cannot identify image file 'G:\\Limit\\COFO\\01 PhD research\\00 Perso\\MRI\\AA242789CJS_1.2.276.0.37.1.373.201611.537590_rnr-vuemotion03_VwfuGt31BUe4lVMJCacX5w.1\\exam\\jpeg\\Thumbs.db'

## 2 — Load the MHD Volume

Opens a file-picker dialog (`tkinter.filedialog.askopenfilename`) to select the `.mhd` file to process, then reads it into memory with `CT_scans.read_mhd_file()`, returning the 3D image array plus its voxel spacing and origin.

In [2]:
#------  select the mhd file -------

# Prompt the user to select a file using a file explorer
root = tk.Tk()
input_file_path = filedialog.askopenfilename() # this is the mhd file contnaing the whole image, no filter, no segmentation
print(f"Input file path is {input_file_path}")
root.withdraw()

file_name = str(input_file_path).split('/')[-1].split('.')[0]


# open the file
working_path = os.path.dirname(input_file_path)
print(f"working path is {working_path}")
file_name = os.path.splitext(os.path.basename(input_file_path))[0]
print(f"file name is {file_name}")


image, spacing, origin = gyroid_utils.CT_scans.read_mhd_file(input_file_path)



2026-07-07 09:24:19,355 - CT_window_logger - WARNING - Please remember to use %matplotlib qt . It tells Matplotlib to open plots in a separate interactive Qt window instead of inline in the notebook.
2026-07-07 09:24:19,359 - CT_window_logger - INFO - starting window setup function


Input file path is C:/Users/cofo/Documents/02 - GitHub/GYROIDS/examples/ball_ct_scan.mhd
working path is C:/Users/cofo/Documents/02 - GitHub/GYROIDS/examples
file name is ball_ct_scan


2026-07-07 09:24:26,122 - CT_window_logger - INFO - window setup function succesfull


## 3 — Extract, Clean, and Export the Surface Mesh

- **Rebuild physical coordinates**: reconstructs x/y/z coordinate grids from the image shape and voxel spacing.
- **`mesh_from_matrix`**: extracts an isosurface mesh from the CT volume via Marching Cubes.
- **`smooth_mesh` → `simplify_mesh` → `smooth_mesh` → `fix_mesh`**: a smooth/simplify/smooth/fix pipeline that turns the raw (and typically huge) marching-cubes output into a clean, manageable mesh.
- **Save**: writes a curvature-colored interactive HTML preview (`viz.save_mesh_as_html`) and exports the mesh as an STL (`mesh_tools.export_as_STL`).

In [ ]:
# convert mhd to stl
#recreate x,y,z axis based on the spacing and origin
z = np.arange(image.shape[2]) * spacing[0]
y = np.arange(image.shape[1]) * spacing[1] 
x = np.arange(image.shape[0]) * spacing[2] 

Y, X, Z = np.meshgrid(y,x,z)
print(X.shape, Y.shape, Z.shape, image.shape)

verts, faces = gyroid_utils.mesh_tools.mesh_from_matrix(
    matrix= image,
    iso_level = 1,
    algo_step_size = 1,
    x = X,
    y = Y,
    z = Z,
    pad_width=10,
    )

from gyroid_utils.logger import set_log_level
set_log_level("DEBUG")

verts, faces = gyroid_utils.mesh_tools.smooth_mesh(verts = verts, faces = faces, smoothing_factor=0.6, iterations=50)
verts, faces = gyroid_utils.mesh_tools.simplify_mesh(verts = verts, faces = faces, target=1000000, mode="open3d")
verts, faces = gyroid_utils.mesh_tools.smooth_mesh(verts = verts, faces = faces, smoothing_factor=0.6, iterations=20)

gyroid_utils.mesh_tools.fix_mesh(verts=verts, faces=faces)

gyroid_utils.viz.save_mesh_as_html(verts = verts, faces = faces, show_curvature_colorscale=True, file_name = str(input_file_path).split('.')[0])
gyroid_utils.mesh_tools.export_as_STL(verts, faces, path = str(input_file_path).split('.')[0] + '.stl')

## 4 — Alternate Visualization (Normal Colorscale)

Re-renders the same mesh as an HTML preview, this time colored by face normals instead of curvature, for a quick visual comparison of the two colorscale options in `viz.save_mesh_as_html()`.

In [4]:
gyroid_utils.viz.save_mesh_as_html(verts = verts, faces = faces, show_normal_colorscale=True, file_name = str(input_file_path).split('.')[0])

[INFO] gyroid_utils:((save_mesh_as_html)): Saving mesh visualization → 'C:/Users/cofo/Documents/02 - GitHub/GYROIDS/examples/ball_ct_scan.html'
[DEBUG] gyroid_utils:((save_mesh_as_html)): Input mesh: 1080 vertices, 2156 faces
[DEBUG] gyroid_utils:((save_mesh_as_html)): Building Plotly figure...
[INFO] gyroid_utils:((save_mesh_as_html)): HTML visualization saved → C:/Users/cofo/Documents/02 - GitHub/GYROIDS/examples/ball_ct_scan.html
